# HUGGE Alba — TRELLIS.2 PBR GLB

Этот notebook бесплатно запускает оптимизированный `trellis.cpp` на GPU Kaggle или Google Colab, загружает фотографию Alba с демонстрационного сайта MIRRAI и сохраняет текстурированную GLB-модель.

В Kaggle откройте **Settings → Accelerator → GPU P100** (или T4) и включите **Internet**. В Colab выберите **Runtime → Change runtime type → T4 GPU**. Затем нажмите **Run All**. Первый запуск скачивает около 9.5 ГБ весов Q8 и поэтому занимает заметное время.

In [ ]:
import os, pathlib, subprocess, time, tarfile, requests

base = pathlib.Path('/kaggle/working') if pathlib.Path('/kaggle/working').exists() else pathlib.Path('/content')
work = base / 'trellis2'
work.mkdir(parents=True, exist_ok=True)
subprocess.run(['nvidia-smi'], check=True)

def remote_size(url):
    response = requests.head(url, allow_redirects=True, timeout=60)
    response.raise_for_status()
    return int(response.headers.get('content-length', 0))

def download_resume(url, destination, retries=8):
    destination = pathlib.Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    expected = remote_size(url)
    if destination.exists() and expected and destination.stat().st_size == expected:
        print(f'✓ {destination.name} уже загружен')
        return
    if destination.exists() and expected and destination.stat().st_size > expected:
        destination.unlink()
    for attempt in range(1, retries + 1):
        offset = destination.stat().st_size if destination.exists() else 0
        headers = {'Range': f'bytes={offset}-'} if offset else {}
        try:
            with requests.get(url, headers=headers, stream=True, allow_redirects=True, timeout=(60, 300)) as response:
                if response.status_code == 416 and expected == offset:
                    return
                response.raise_for_status()
                append = offset > 0 and response.status_code == 206
                mode = 'ab' if append else 'wb'
                if not append:
                    offset = 0
                downloaded = offset
                last_report = downloaded
                with destination.open(mode) as target:
                    for chunk in response.iter_content(8 * 1024 * 1024):
                        if chunk:
                            target.write(chunk)
                            downloaded += len(chunk)
                            if downloaded - last_report >= 512 * 1024 * 1024:
                                print(f'  {destination.name}: {downloaded / 1024**3:.1f} / {expected / 1024**3:.1f} ГБ')
                                last_report = downloaded
            if not expected or destination.stat().st_size == expected:
                print(f'✓ {destination.name}')
                return
            raise IOError(f'неполный файл: {destination.stat().st_size} из {expected}')
        except Exception as error:
            if attempt == retries:
                raise
            print(f'Повтор {attempt}/{retries}: {error}')
            time.sleep(min(30, attempt * 3))

import socket

runtime = work / 'runtime'
server_bin = runtime / 'trellis-server'
if not server_bin.exists():
    compute_cap = subprocess.check_output(['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'], text=True).strip().splitlines()[0]
    backend = 'cuda12' if float(compute_cap) < 7.5 else 'cuda'
    archive = work / f'trellis-{backend}-linux-x64.tar.gz'
    download_resume(f'https://github.com/pwilkin/trellis.cpp/releases/latest/download/trellis-{backend}-linux-x64.tar.gz', archive)
    runtime.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive, 'r:gz') as bundle:
        bundle.extractall(runtime)
    server_bin.chmod(0o755)

models = work / 'models'
model_names = ['birefnet.gguf', 'dinov3.gguf', 'ss_flow.gguf', 'ss_dec.gguf', 'shape_flow_512.gguf', 'shape_flow_1024.gguf', 'shape_dec.gguf', 'tex_flow_512.gguf', 'tex_flow_1024.gguf', 'tex_dec.gguf']
for model_name in model_names:
    download_resume(f'https://huggingface.co/ilintar/trellis2-gguf/resolve/main/q8/{model_name}', models / model_name)
print('TRELLIS.2 runtime and Q8 weights are ready')

In [ ]:
runtime = work / 'runtime'
server_bin = runtime / 'trellis-server'
models = work / 'models'
log_path = work / 'trellis-server.log'
previous_server = globals().get('server')
previous_url = globals().get('server_url')
already_running = previous_server is not None and previous_server.poll() is None and previous_url
if already_running:
    try:
        health = requests.get(f'{previous_url}/health', timeout=3)
        already_running = health.ok and health.text.strip() == 'ok'
    except requests.RequestException:
        already_running = False
if already_running:
    server_url = previous_url
    print('TRELLIS.2 server уже запущен — используем его повторно')
else:
    with socket.socket() as free_socket:
        free_socket.bind(('127.0.0.1', 0))
        server_port = free_socket.getsockname()[1]
    server_url = f'http://127.0.0.1:{server_port}'
    env = os.environ.copy()
    env['LD_LIBRARY_PATH'] = f"{runtime}:{env.get('LD_LIBRARY_PATH', '')}"
    log = open(log_path, 'w')
    server = subprocess.Popen([str(server_bin), '--models', str(models), '--host', '127.0.0.1', '--port', str(server_port), '--res', '1024', '--require-gpu'], stdout=log, stderr=subprocess.STDOUT, env=env)
    for _ in range(180):
        try:
            if requests.get(f'{server_url}/health', timeout=2).ok:
                break
        except requests.RequestException:
            pass
        if server.poll() is not None:
            raise RuntimeError(log_path.read_text()[-5000:])
        time.sleep(2)
    else:
        raise TimeoutError('TRELLIS server did not become ready')
print('TRELLIS.2 server is ready')

In [ ]:
image_url = 'https://mirrai-try-on.moonlight-5782.chatgpt.site/catalog-sources/hugge-md/alba-89990-1.jpg'
image_path = work / 'alba-89990-1.jpg'
image_response = requests.get(image_url, timeout=60)
image_response.raise_for_status()
image_path.write_bytes(image_response.content)
output = base / 'alba-trellis2-pbr.glb'
with image_path.open('rb') as source:
    response = requests.post(
        f'{server_url}/generate',
        files={'image': ('alba.jpg', source, 'image/jpeg')},
        data={'seed': '89990', 'resolution': '1024', 'bg_removal': 'birefnet'},
        timeout=3600,
    )
if not response.ok:
    raise RuntimeError(f'TRELLIS HTTP {response.status_code}: {response.text[:3000]}')
output.write_bytes(response.content)
print(f'Готово: {output} ({output.stat().st_size / 1024 / 1024:.1f} MB)')
from IPython.display import FileLink, display
display(FileLink(str(output)))
try:
    from google.colab import files
    files.download(str(output))
except ImportError:
    pass